# 머신러닝 v1 — 수치 변경과 모델 비교

**첫 설정 셀을 수정한 뒤 ‘모두 실행’하세요.**

- `MODE = "evaluate"`: 저장된 예측으로 경보 비율만 바꿔 비교합니다. 재학습하지 않습니다.
- `MODE = "train"`: 입력 기간·예측 기간·컬럼·모델 설정을 바꿔 4개 모델을 다시 학습합니다.
- 기본 화면은 **검증 구간**입니다. 최종 평가도 보려면 `SHOW_TEST = True`로 바꾸세요.

경보 비율만 바꾸면 기존 예측을 재사용합니다. 학습 설정이 바뀌면 예전 예측으로 평가하지 않고 재학습을 안내합니다. 새 학습 결과는 설정별 하위 폴더에 저장하며 최초 실험 결과는 보존합니다.


In [1]:
# ===== 여기의 수치를 바꾸세요 =====
MODE = "evaluate"                 # "evaluate": 재평가 / "train": 재학습
SHOW_TEST = False                 # 최종 평가까지 표시하려면 True

ALERT_PERCENTAGES = [0.1, 0.5, 1.0]  # 단위: %. 1.0은 매일 상위 1%
PRIMARY_ALERT_PERCENT = 1.0        # 검증 구간에서 모델을 고를 경보 비율

INPUT_DAYS = 30                   # 참고할 과거 기록 기간
PREDICT_DAYS = 30                 # 이후 며칠 안의 고장을 예측할지
SMART_COLUMNS = [5, 187, 188, 197] # 사용할 SMART raw 번호
DELTA_DAYS = [7, 30]              # 변화량 기간. INPUT_DAYS 이하여야 함
NEGATIVE_RATIO = 20               # 학습 정상:고장 비율 최대 20:1

TRAIN_PERIOD = ("2024-04-01", "2025-12-31")
VALIDATION_PERIOD = ("2026-01-01", "2026-03-31")
TEST_PERIOD = ("2026-04-01", "2026-06-30")
# 각 구간의 마지막 PREDICT_DAYS일은 정답 확인에만 사용합니다.

SEED = 42
CPU_JOBS = 2                     # 동시에 사용할 CPU 작업 수
XGBOOST_DEVICE = "auto"          # auto: GPU 확인 / cpu: CPU 고정
LIGHTGBM_DEVICE = "auto"         # 현재 설치 환경에서는 CPU로 실행됨

LOGISTIC_PARAMS = {"C": 1.0, "max_iter": 1000}
RF_PARAMS = {"n_estimators": 150, "max_depth": 14, "min_samples_leaf": 10}
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 6, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8,
}
LGBM_PARAMS = {
    "n_estimators": 300, "num_leaves": 31, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8,
}


## 1. 설정 확인

재할당 섹터(5)·보고된 복구 불가 오류(187)·보류 중인 섹터(197)를 기본 후보로 사용합니다. 명령 시간 초과(188)는 복합 인코딩 가능성이 있어 **현재 값·0 초과 여부만** 사용합니다. 다른 번호를 추가할 때는 [EDA v1](../../eda/eda_v1_5%25.ipynb)과 원본 열 설명을 확인하세요.

입력에는 예측 시점까지의 기록만 사용합니다. 구간 끝에서 정답 기간을 자동으로 빼서 학습·검증·평가의 미래 정답이 겹치지 않게 합니다.


In [2]:
from pathlib import Path
from copy import deepcopy
import sys, importlib, json
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown

# 저장소 최상위 또는 ml/v1에서 실행할 수 있습니다.
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "ml/v1/train.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("이 저장소 안에서 노트북을 실행하세요.")
FOLDER = ROOT / "ml/v1"
SOURCE = ROOT / "data/backblaze/merged/st12000nm0008_2024Q2-2026Q2.parquet"
sys.path.insert(0, str(FOLDER)) if str(FOLDER) not in sys.path else None
import train, notebook_runner
importlib.reload(train)
importlib.reload(notebook_runner)

training = deepcopy(notebook_runner.DEFAULT)
training.update(input_days=INPUT_DAYS, horizon_days=PREDICT_DAYS,
    smart_columns=SMART_COLUMNS, delta_days=DELTA_DAYS, negative_ratio=NEGATIVE_RATIO,
    seed=SEED, n_jobs=CPU_JOBS, xgboost_device=XGBOOST_DEVICE,
    lightgbm_device=LIGHTGBM_DEVICE,
    periods={"train": TRAIN_PERIOD, "validation": VALIDATION_PERIOD, "test": TEST_PERIOD},
    model_params={"logistic_regression": LOGISTIC_PARAMS, "random_forest": RF_PARAMS,
                  "xgboost": XGB_PARAMS, "lightgbm": LGBM_PARAMS})
settings = notebook_runner.normalize(training)
if PRIMARY_ALERT_PERCENT not in ALERT_PERCENTAGES:
    raise ValueError("PRIMARY_ALERT_PERCENT를 ALERT_PERCENTAGES에도 넣어주세요.")
if any(not 0 < float(p) <= 100 for p in ALERT_PERCENTAGES):
    raise ValueError("경보 비율은 0 초과~100 이하로 입력하세요.")
display(pd.DataFrame([
    {"구간": k, "예측 시작일": v[0], "예측 마지막 날": v[1],
     "정답 확인 마지막 날": settings["periods"][k][1]}
    for k, v in settings["splits"].items()]))
print(f"참고 {INPUT_DAYS}일 → 이후 {PREDICT_DAYS}일 고장 예측 / SMART {SMART_COLUMNS}")


참고 30일 → 이후 30일 고장 예측 / SMART [5, 187, 188, 197]


,구간,예측 시작일,예측 마지막 날,정답 확인 마지막 날
0,train,2024-04-01,2025-12-01,2025-12-31
1,validation,2026-01-01,2026-03-01,2026-03-31
2,test,2026-04-01,2026-05-31,2026-06-30


## 2. 실행

`evaluate`에서는 아래 셀이 저장된 예측을 찾기만 합니다. `train`에서는 피처 생성부터 4개 모델 학습까지 실행합니다. 기간·컬럼을 바꾸면 수 분 이상 걸릴 수 있습니다.


In [3]:
RUN = notebook_runner.resolve_run(FOLDER, SOURCE, settings, MODE)
print("사용한 결과:", RUN.relative_to(ROOT))
audit = json.loads((RUN / "data_audit.json").read_text(encoding="utf-8"))
data_summary = pd.DataFrame(audit["splits"]).T
data_summary["고장 예정 행 비율(%)"] = 100 * data_summary["positives"].astype(float) / data_summary["rows"].astype(float)
display(data_summary[["rows", "positives", "disks", "positive_disks", "고장 예정 행 비율(%)"]].rename(
    columns={"rows":"평가 가능 행", "positives":"고장 예정 행", "disks":"디스크 수", "positive_disks":"고장 예정 디스크 수"}))


사용한 결과: ml\v1\artifacts


,평가 가능 행,고장 예정 행,디스크 수,고장 예정 디스크 수,고장 예정 행 비율(%)
train,11025886,22288.0,19473,801,0.202142
validation,1110676,2127.0,18634,119,0.191505
test,1121321,2384.0,18516,144,0.212606


## 3. 경보 비율별 비교

| 항목 | 읽는 법 |
|---|---|
| 정밀도 | 경보 중 설정한 미래 기간 안에 실제로 고장 난 비율. 높을수록 헛경보가 적음 |
| 재현율 | 실제 고장 예정 행 중 경보로 잡은 비율. 높을수록 놓치는 고장이 적음 |
| 고장 디스크 탐지율 | 고장 예정 디스크 중 고장 전 예측 기간 안에 한 번이라도 경보한 비율 |
| 일평균 헛경보 수 | 해당 미래 기간 안에 고장 나지 않은 대상에 낸 경보 수의 하루 평균 |
| PR-AUC(AP) | 위험 순위 전체의 성능. 정확도와는 다른 점수 |

**하루에 같은 디스크를 다시 경보하면 날짜마다 별도 건으로 셉니다.** 헛경보는 미래 기간 안에 고장 나지 않았다는 뜻이며, 영구 정상이나 실제 헛교체를 뜻하지 않습니다. 정상 축소 학습의 점수는 보정된 고장 확률이 아닙니다.


In [4]:
comparison, baseline, selected = notebook_runner.evaluate_run(
    RUN, ALERT_PERCENTAGES, PRIMARY_ALERT_PERCENT, SHOW_TEST)
columns = ["모델", "경보 비율(%)", "PR-AUC(AP)", "정밀도(%)", "재현율(%)",
           "고장 디스크 탐지율(%)", "일평균 경보 수", "일평균 헛경보 수"]
for split in comparison["구간"].unique():
    display(Markdown("### " + ("검증 결과" if split == "validation" else "최종 평가 결과")))
    display(comparison.loc[comparison["구간"] == split, columns].round(3).reset_index(drop=True))
display(Markdown("**SMART 규칙의 원래 경보량과 성능** — 경보 수가 다르면 정밀도만으로 우열을 단정하지 않습니다."))
display(baseline.round(3))
print(f"검증 구간 상위 {PRIMARY_ALERT_PERCENT:g}% 정밀도로 선택: {notebook_runner.LABELS[selected]}")


검증 구간 상위 1% 정밀도로 선택: XGBoost


### 검증 결과

,모델,경보 비율(%),PR-AUC(AP),정밀도(%),재현율(%),고장 디스크 탐지율(%),일평균 경보 수,일평균 헛경보 수
0,로지스틱 회귀,0.1,0.204,35.614,19.088,44.538,19.00,12.233
1,로지스틱 회귀,0.5,0.204,19.875,52.139,73.950,93.00,74.517
2,로지스틱 회귀,1.0,0.204,12.111,63.423,80.672,185.65,163.167
3,Random Forest,0.1,0.252,43.772,23.460,43.697,19.00,10.683
4,Random Forest,0.5,0.252,20.591,54.020,70.588,93.00,73.850
5,Random Forest,1.0,0.252,12.362,64.739,81.513,185.65,162.700
6,XGBoost,0.1,0.306,53.947,28.914,47.899,19.00,8.750
7,XGBoost,0.5,0.306,21.022,55.148,77.311,93.00,73.450
8,XGBoost,1.0,0.306,12.524,65.585,84.034,185.65,162.400
9,LightGBM,0.1,0.308,51.491,27.598,45.378,19.00,9.217


**SMART 규칙의 원래 경보량과 성능** — 경보 수가 다르면 정밀도만으로 우열을 단정하지 않습니다.

,구간,방식,정밀도(%),재현율(%),고장 디스크 탐지율(%),일평균 경보 수,일평균 헛경보 수,경보 건수,실제 고장으로 이어진 경보,헛경보 건수,원래 고장 비율(%)
0,validation,SMART 규칙,1.76,78.326,87.395,1577.883,1550.117,94673,1666,93007,0.192


## 4. 경보를 줄이면 어떻게 달라지나

위쪽일수록 정밀도가 높고, 오른쪽일수록 고장을 더 많이 잡습니다. 점에 마우스를 올리면 경보 비율과 경보 수를 볼 수 있습니다. 비교와 설정 선택은 기본적으로 검증 결과를 사용합니다.


In [5]:
pio.renderers.default = "plotly_mimetype"
view = comparison[comparison["구간"] == "validation"].copy()
view["경보 범위"] = view["경보 비율(%)"].map(lambda x: f"상위 {x:g}%")
fig = px.scatter(view.sort_values("경보 비율(%)"), x="재현율(%)", y="정밀도(%)",
    color="모델", hover_data=["경보 범위", "일평균 경보 수", "일평균 헛경보 수"],
    title="검증 구간: 경보 범위별 정밀도와 재현율", template="plotly_white")
fig.update_traces(mode="lines+markers", marker_size=10)
fig.update_layout(font_family="Malgun Gothic", height=520, margin=dict(t=80))
fig.show()

work = view[view["경보 비율(%)"] == PRIMARY_ALERT_PERCENT].copy()
work["고장으로 이어진 경보"] = work["일평균 경보 수"] - work["일평균 헛경보 수"]
chart = work.melt(id_vars="모델", value_vars=["고장으로 이어진 경보", "일평균 헛경보 수"],
                  var_name="경보 결과", value_name="하루 평균 건수")
fig2 = px.bar(chart, x="모델", y="하루 평균 건수", color="경보 결과", barmode="stack",
    title=f"검증 구간: 매일 상위 {PRIMARY_ALERT_PERCENT:g}% 경보의 구성", template="plotly_white",
    color_discrete_map={"고장으로 이어진 경보":"#2a78d6", "일평균 헛경보 수":"#eb6834"})
fig2.update_layout(font_family="Malgun Gothic", height=440)
fig2.show()


## 5. 결과 저장 — 주석 처리

아래 비교표·평가 설정 저장 코드는 실행하지 않습니다. 결과는 위의 표·그래프로 확인하세요.

이 셀은 별도 CSV·평가 설정 저장만 제어합니다. `MODE = "train"`의 학습 모델·피처·예측 파일 저장은 학습 코드에서 별도로 수행합니다.


In [ ]:
# SAVED = notebook_runner.save_evaluation(RUN, comparison, baseline, selected,
#     ALERT_PERCENTAGES, PRIMARY_ALERT_PERCENT, SHOW_TEST)
# print("비교표:", (SAVED / "comparison.csv").relative_to(ROOT))
# print("설정:", (SAVED / "evaluation_config.json").relative_to(ROOT))